# CSE476 Agentic AI and Intelligent Automation
## CA1 Project 1: Build a Real Agent (Autonomous Problem Solver)

**Student Name:** Anmol Nagpal  
**Course:** CSE476 Agentic AI and Intelligent Automation  
**Submission Type:** Solo  
**Maximum Marks:** 30  

---

### 🎯 The Core Requirement: An Agent, Not a Chatbot
> *"A chatbot is a vending machine, one question in, one answer out. An agent is an intern, you give it a goal and it decides which tools to use, does several steps, remembers what it found, and comes back with a result."*

This notebook demonstrates all three requirements of the official rubric:
1. **Tool Invocation:** Calls at least two tools (`fetch_problem_spec`, `execute_code_sandbox`, `explain_algorithm_complexity`).
2. **Multi-Step Plan-Act Loop:** Decides next steps dynamically based on intermediate tool results.
3. **Memory:** Retains state across turns and utilizes prior dialogue context in subsequent decisions.


In [ ]:
# Cell 1: Imports and Agent Initialization
import sys
import os

# Add current directory to path
sys.path.append('.')

from memory import AgentMemory
from tools import AGENT_TOOLS, fetch_problem_spec, execute_code_sandbox, explain_algorithm_complexity
from agent import LeetCodeAssistantAgent

print("Initializing Autonomous Coding Agent with ReAct Plan-Act Loop & Memory...")
agent = LeetCodeAssistantAgent()
print("Agent ready! Loaded Tools:", list(agent.tools.keys()))


---
### 🧪 Goal 1: Multi-Step Autonomous Problem Solving with Sandbox Verification
**Goal:** `"Help me solve Two Sum, write the code and verify it against tests"`

**Expected Agent Behavior:**
1. **Step 1:** Formulate plan and call `fetch_problem_spec` to retrieve requirements and test cases.
2. **Step 2:** Formulate solution and call `execute_code_sandbox` to run test assertions.
3. **Step 3:** Call `explain_algorithm_complexity` to compute Big-O asymptotic bounds.
4. **Step 4:** Return final verified answer and store problem details in working memory.


In [ ]:
# Cell 2: Run Goal 1
goal_1 = "Help me solve Two Sum, write the code and verify it against tests"
print(f"Goal: {goal_1}\n")
res_1 = agent.run(goal_1)

print("=" * 60)
print("PLAN-ACT EXECUTION TRACE:")
print("=" * 60)
for t in res_1["traces"]:
    print(f"\n>> [STEP {t.step_num}]")
    print(f"  [Thought/Plan] : {t.thought}")
    print(f"  [Action (Tool)]: {t.action}")
    print(f"  [Action Input] : {t.action_input}")
    obs_preview = str(t.observation)
    if len(obs_preview) > 140:
        obs_preview = obs_preview[:140] + "... [truncated]"
    print(f"  [Observation]  : {obs_preview}")

print("\n" + "=" * 60)
print("FINAL AGENT OUTPUT:")
print("=" * 60)
print(res_1["final_answer"])


---
### 🧠 Goal 2: Multi-Turn Conversational Memory & Context Retention
**Goal:** `"Now recommend a follow-up problem using a similar concept in C++"`

**Expected Agent Behavior:**
1. **Memory Consultation:** The agent accesses `AgentMemory` to recall that Turn 1 addressed **Two Sum** using a **Hash Map** in Python.
2. **Contextual Resolution:** Recommends **Group Anagrams** as the natural follow-up problem using hash maps.
3. **State Transition:** Switches target language to `C++` as instructed, executes sandbox verification, and references the previous turn in its response.


In [ ]:
# Cell 3: Run Goal 2 (Testing Memory)
goal_2 = "Now recommend a follow-up problem using a similar concept in C++"
print(f"Goal: {goal_2}\n")
res_2 = agent.run(goal_2)

print("=" * 60)
print("PLAN-ACT EXECUTION TRACE:")
print("=" * 60)
for t in res_2["traces"]:
    print(f"\n>> [STEP {t.step_num}]")
    print(f"  [Thought/Plan] : {t.thought}")
    print(f"  [Action (Tool)]: {t.action}")
    print(f"  [Action Input] : {t.action_input}")
    obs_preview = str(t.observation)
    if len(obs_preview) > 140:
        obs_preview = obs_preview[:140] + "... [truncated]"
    print(f"  [Observation]  : {obs_preview}")

print("\n" + "=" * 60)
print("FINAL AGENT OUTPUT:")
print("=" * 60)
print(res_2["final_answer"])


---
### 🔄 Goal 3: Self-Healing Error Recovery (Handling Tool Failure)
**Goal:** `"Solve problem 1, but simulate a buggy initial submission to test error recovery"`

**Expected Agent Behavior:**
1. **Step 1:** Problem fetched.
2. **Step 2:** Initial candidate code is executed in sandbox and **FAILS** with a `Wrong Answer` verdict on non-adjacent elements.
3. **Step 3 (The Agent Decides Next Step):** Rather than giving up or outputting the error, the agent's Plan-Act loop inspects the failure observation, diagnoses the bug, replaces the naive logic with a Hash Map, and **re-runs the sandbox tool**.
4. **Step 4:** Code passes all test cases, and the agent completes the goal successfully.


In [ ]:
# Cell 4: Run Goal 3 (Self-Healing Recovery)
goal_3 = "Solve problem 1, but simulate a buggy initial submission to test error recovery"
print(f"Goal: {goal_3}\n")
res_3 = agent.run(goal_3, simulate_initial_failure=True)

print("=" * 60)
print("PLAN-ACT EXECUTION TRACE:")
print("=" * 60)
for t in res_3["traces"]:
    print(f"\n>> [STEP {t.step_num}]")
    print(f"  [Thought/Plan] : {t.thought}")
    print(f"  [Action (Tool)]: {t.action}")
    print(f"  [Action Input] : {t.action_input}")
    obs_preview = str(t.observation)
    if len(obs_preview) > 140:
        obs_preview = obs_preview[:140] + "... [truncated]"
    print(f"  [Observation]  : {obs_preview}")

print("\n" + "=" * 60)
print("FINAL AGENT OUTPUT:")
print("=" * 60)
print(res_3["final_answer"])


---
### 🔍 Memory Audit: Verifying State Retention Across Turns
Let's inspect `agent.memory` to prove persistent conversational memory.


In [ ]:
# Cell 5: Inspect Memory
print(f"Total Stored Conversation Turns: {len(agent.memory.turns)}\n")
print("Recent History Summary:")
print(agent.memory.get_recent_history())
print("\nWorking State Keys:")
for k, v in agent.memory.working_state.items():
    print(f"  • {k}: {v}")


---
## 🎓 Viva Preparation & Code Reference Guide

| Question | Code Location | Key Explanation |
|---|---|---|
| **1. Where the plan-act loop decides the next step?** | `agent.py:lines 185-260` | The `run()` method initiates a multi-step sequence. At each step, it generates a `thought`, calls a tool, checks the `observation`, and branches dynamically (e.g. if `status != 'PASSED'`, it enters the self-healing retry block). |
| **2. Walk through a tool call?** | `agent.py:lines 211-220` and `tools.py:lines 140-230` | In `agent.py`, the agent builds `tool_args` and invokes `self.tools['execute_code_sandbox']['fn'](**tool_args)`. `tools.py` compiles the code inside an isolated dictionary scope and evaluates test inputs. |
| **3. Show where memory is read back?** | `agent.py:lines 170-175` and `memory.py:lines 65-80` | At the beginning of `run()`, `self.memory.get_recent_history()` and `self.memory.get('last_problem_id')` are read to resolve pronoun references (`'follow-up problem'`) before tool selection. |
